<a href="https://colab.research.google.com/github/tsukki8/ds2002-fa26/blob/main/notebooks/05-cleaning-gauntlet/2026-09-25%20%E2%80%94%20Cleaning%20Gauntlet%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [2]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [3]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [4]:
# TODO
duplicates = df.duplicated().sum()
df = df.drop_duplicates().copy()

log("duplicates", "Dropped exact duplicate rows", duplicates)
print("Rows remaining:", len(df))

[duplicates] Dropped exact duplicate rows (15 row(s))
Rows remaining: 300


### TODO 2 — clean `price` -> float

In [5]:
# TODO
df["price"] = (
    df["price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.strip()
    .astype(float)
)

log("price", "Removed dollar signs and converted prices to float", len(df))
print(df["price"].dtype)

[price] Removed dollar signs and converted prices to float (300 row(s))
float64


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [6]:
# TODO
df["qty"] = pd.to_numeric(df["qty"], errors="coerce")

missing_qty = df["qty"].isna().sum()
negative_qty = (df["qty"] < 0).sum()

df = df.dropna(subset=["qty"]).copy()
log("missing_qty", "Dropped rows with missing quantities", missing_qty)

df = df[df["qty"] > 0].copy()
log("negative_qty", "Dropped rows with negative quantities", negative_qty)

print("Rows remaining:", len(df))

[missing_qty] Dropped rows with missing quantities (12 row(s))
[negative_qty] Dropped rows with negative quantities (13 row(s))
Rows remaining: 275


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [7]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
print(df["item"].value_counts())

ITEM_MAP = {
    "Cheeseburger": "Cheeseburger",
    "cheese burger": "Cheeseburger",
    "Foam Finger": "Foam Finger",
    "foam finger": "Foam Finger",
    "Rain Poncho": "Rain Poncho",
    "rain poncho": "Rain Poncho"
}

variants_before = df["item"].nunique()
df["item"] = df["item"].map(ITEM_MAP)
variants_after = df["item"].nunique()

log(
    "item",
    "Combined six item spellings into three canonical products",
    variants_before - variants_after
)

print(df["item"].value_counts())

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] Combined six item spellings into three canonical products (3 row(s))
item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [8]:
# TODO
print(df["category"].value_counts())

CATEGORY_MAP = {
    "Food": "Food",
    "food": "Food",
    "Merch": "Merch",
    "Apparel": "Merch",
    "RainGear": "RainGear",
    "rain-gear": "RainGear"
}

categories_before = df["category"].nunique()
df["category"] = df["category"].map(CATEGORY_MAP)
categories_after = df["category"].nunique()

log(
    "category",
    "Normalized spelling and grouped Apparel with Merch",
    categories_before - categories_after
)

print(df["category"].value_counts())

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[category] Normalized spelling and grouped Apparel with Merch (3 row(s))
category
Food        95
Merch       94
RainGear    86
Name: count, dtype: int64


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [13]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
# TODO: assert something about item
# TODO: assert something about category
print('clean:', df.shape)

assert set(df["item"].unique()) == {
    "Cheeseburger", "Foam Finger", "Rain Poncho"
}

assert set(df["category"].unique()) == {
    "Food", "Merch", "RainGear"
}

print("All assertions passed!")
print("Clean dataset shape:", df.shape)

clean: (275, 6)
All assertions passed!
Clean dataset shape: (275, 6)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [10]:
# TODO
df["revenue"] = df["qty"] * df["price"]

revenue_by_category = (
    df.groupby("category")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .round(2)
)

print(revenue_by_category)
print(f"\nOverall revenue: ${df['revenue'].sum():,.2f}")

category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64

Overall revenue: $4,740.00


**What I would tell the vendor:** I would recommend selling food because that category had the highest revenue of $1656.0 on average.

### TODO 8 — read back your log

In [11]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,Dropped exact duplicate rows,15
1,price,Removed dollar signs and converted prices to f...,300
2,missing_qty,Dropped rows with missing quantities,12
3,negative_qty,Dropped rows with negative quantities,13
4,item,Combined six item spellings into three canonic...,3
5,category,Normalized spelling and grouped Apparel with M...,3


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

**a)** Removing the 15 rows of exact duplicate changes the revenue from $4,852.50 to $4,594.50, decreasing a total of $258.00.

**b)** I chose to remove the negative quantities in step 3, while another person might be choosing to keep them as refunds stored as a separate column and calculate the net revenue instead of removing the negatives completely from the dataset.